<a href="https://colab.research.google.com/github/DanielleGroenewegen/ChickenHotWings/blob/main/working_fileGroup32_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Do LLMs know the difference between a pet chicken and a roast chicken?

## Word sense disambiguation in computational models and humans


In human language, words do not always have a fixed meaning. The most striking example is homonymous words: words that have the same form, but very different meanings. For instance, the word "bank", which has a different meaning in the context "I went to the bank to get some money" and "At the river bank, I met my old friend". Polysemous words are words that have different -- yet related -- meanings: for example, "chicken" is the same 'entity' in "My pet chicken is lovely" and "I am having roast chicken for dinner", but has very different meanings in these two contexts. In general, context can modulate almost any word's meaning. This poses a challenge in computational linguistics, as we need to find a way to differentiate among different meanings like humans do. Much research, resources, and models have been put forward to help with this challenge.

In this assignment, you are going to focus on [Trott and Bergen's (2021)](https://aclanthology.org/2021.acl-long.550/) RAW-C dataset: you are going to conduct a number of explorations with this dataset and partially replicate their research by the end of the assignment. In short, the authors explore how good LLMs are at capturing same/different meanings of words across contexts by comparing it to human judgements. To better understand the idea and the research, start by reading the paper.

This assignment entails a series of (interconnected) tasks (altogether worth 95 points):

* **Task 1**. Compute contextual word embeddings at different layers from Trott & Bergen's dataset. Here, each word is found in 4 sentences: 2 with one meaning, 2 with another meaning.
* **Task 2**. Compute sense embeddings for words in Trott & Bergen's dataset using WordNet, so you have an embedding for each definition of the word.
* **Task 3**. Compute the similarity between the contextual word embeddings of the homonyms at different layers and their sense embeddings; explore the relationship between homonyms and dominant senses quantitatively and qualitatively
* **Task 4**. Replicate part of Trott & Bergen's work by computing similarities across sentences with same/different meanings at the different layers and correlate with human similarities; visualise the results and reflect on them

In order to better understand the assignment, we recommend going through it all before starting so that it is clear how each part is connected to the next (which will help you make decisions about data structures, for instance).

# Task 1: Compute contextual word embeddings for homonyms [20 points]

## Task 1.1: read, explore and extract the necessary data [5 points]

First, you will have to (fork and) clone the github repository that stores the data you'll need. This can be found here: https://github.com/sashakenjeeva/raw-c . The repo also includes a README with a description of the original files in the repository, as well as some notes relevant for this assignment specifically.

In [ ]:
#your code here (you can use as many cells as necessary/you prefer)

Make sure you mount the drive now so that you have access to the folder (think about setting the working directory in a way that is convenient).

In [1]:
# mount the drive here
!git clone https://github.com/DanielleGroenewegen/ChickenHotWings.git


Cloning into 'ChickenHotWings'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 133 (delta 44), reused 89 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 7.67 MiB | 7.34 MiB/s, done.
Resolving deltas: 100% (44/44), done.


Now, you will have to read the data and organise it in a structure that works for the next parts of the assignment.

Read and explore the dataframe to see its structure (print part of it). What we need from it are the homonyms (in the form that they appear in the sentence -- the lexeme -- and in their regular form -- the lemma) and their corresponding sentences with different meanings (M1_a and M1_b have same meaning; M2_a, M2_b have same meaning). We only will need the stimuli that are in the final RAW-C dataset, as this is what we'll replicate at the end.

You can decide which data structure to use, but make sure that all these pieces of information are there (the word, the string, the meaning id, and the corresponding sentences) and easy to retrieve. Show your data at the end, as well as how many stimuli you end up with.

In [2]:
# read the data here
import pandas as pd
sunflower = pd.read_csv("/content/ChickenHotWings/data/stims/stimuli.csv")
#print(sunflower.head())
data = sunflower[['String', 'Word','M1_a', 'M1_b', 'M2_a', 'M2_b']].copy()

# remember to print the data structure you produce and the number of stimuli it contains!
print(data.head(),"\n")
#print(data.tail())
print(f'Number of stimuli: {len(data)}')

      String       Word                            M1_a  \
0       lamb       lamb  They liked the marinated lamb.   
1       book       book     He had a best-selling book.   
2  breakfast  breakfast   They ate a pancake breakfast.   
3    chicken    chicken   They liked the juicy chicken.   
4      class      class      It was a perceptive class.   

                               M1_b                              M2_a  \
0      They liked the grilled lamb.         They liked the cute lamb.   
1            He had a popular book.              He had a heavy book.   
2  They ate a nutritious breakfast.      They ate a family breakfast.   
3   They liked the roasted chicken.  They liked the clucking chicken.   
4       It was a misbehaving class.            It was a boring class.   

                            M2_b  
0  They liked the friendly lamb.  
1   He had a leather-bound book.  
2   They ate a lonely breakfast.  
3    They liked the pet chicken.  
4    It was a stimulating class

## Task 1.2: Compute the contextualised word embeddings [15 points]


Now that you have the homonyms and their corresponding sentences, we will need to compute word embeddings for each of them. For this we will use the BERT base model, in its uncased version.

That is, for each homonym, you will have to compute four embeddings: one for the homonym in M1_a, one in M1_b, one in M2_a, one in M2_b. However, we also want to look into different layers of the BERT model to see which one captures the homonym's meaning best: you want to calculate embeddings at the static layer and at layers 4, 8, 12.

We will use the package psycho-embeddings (you will use it in class), which allows us to specify which target words we want to obtain the embeddings of, in which sentences, and at which layers, among other things. Make sure to read the documentation of the package so that you know the meaning of the arguments and which ones will come useful to you.

First of all, install the psycho-embeddings package below.

Now, import the relevant module/function from psycho-embeddings and load the required BERT model.

In [3]:
# install the psycho-embeddings package here

!git clone https://github.com/MilaNLProc/psycho-embeddings.git
%cd /content/psycho-embeddings
!pip install -e . #current folder


Cloning into 'psycho-embeddings'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 199 (delta 105), reused 141 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (199/199), 67.91 KiB | 7.55 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/content/psycho-embeddings
Obtaining file:///content/psycho-embeddings
  Preparing metadata (setup.py) ... done
  Running setup.py develop for psycho_embeddings


In [4]:
#your code here

from psycho_embeddings import ContextualizedEmbedder
model = ContextualizedEmbedder("bert-base-cased", max_length=128)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--bert-base-cased/snapshots/cd5ef92a9fb2f889e972770a36d4ed042daf221e/config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_hidden_states": true,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--bert-base-cased/snapshots/cd5ef92a9fb2f889e972770a36d4ed042daf221e/model.safetensors
Since the `dtype` attribute can't be found in model's config object, will use dtype={dtype} as derived from model's weights


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--bert-base-cased/snapshots/cd5ef92a9fb2f889e972770a36d4ed042daf221e/config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Now, test that everything works correctly by computing an embedding for the word "assignment" in the sentence "I am having so much fun with this assignment!", at static layer and layers 4, 8 and 12 (hint: think of tokenisation and how the embedder deals with that).

In [5]:
#your code here
embeddings = model.embed(
    words=["assignment"],
    target_texts=["I am having so much fun with this assignment!"],
    layers_id= [4, 8, 12],
    batch_size=1,
    return_static=True, #static layer
)




Text tokenization:   0%|          | 0/1 [00:00<?, ? examples/s]

  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 1/1 [00:00<00:00,  1.51it/s]


The next step is to calculate embeddings for the homonyms and their sentences that we got from the RAW-C dataset.

Make sure that your final output includes the word, the meaning id (M1_a, etc), the corresponding sentence and the embeddings at static layer and layers 4, 8, 12. You should maximally optimise this process by calculating in batches (again, check psycho-embeddings documentation), but keep in mind this might still take a while. First test your pipeline with a small number of inputs, and only run the full scale embedding extraction once you're positive the code works as expected.

When done, save the output in [pickle](https://docs.python.org/3/library/pickle.html) format (this is similar to json, but it can also handle np.arrays), so that you can easily load it later when needed and do not have to run it again. After pickle dumping (that's the word for saving it in pickle format), print it so that you are sure everything was saved correctly.

Then, check that your final data includes everything that you need by checking the entry "bank" and print the data pertaining to "bank".

In [6]:
items = []

meaning_cols = ["M1_a", "M1_b", "M2_a", "M2_b"]

for _, row in data.iterrows():
    for col in meaning_cols:
        if pd.notna(row[col]):
            items.append({
                "Word": row["Word"],   # use capital W everywhere
                "meaning_id": col,
                "sentence": row[col]
            })

print("Total items:", len(items))
print(items[:3])

Total items: 460
[{'Word': 'lamb', 'meaning_id': 'M1_a', 'sentence': 'They liked the marinated lamb.'}, {'Word': 'lamb', 'meaning_id': 'M1_b', 'sentence': 'They liked the grilled lamb.'}, {'Word': 'lamb', 'meaning_id': 'M2_a', 'sentence': 'They liked the cute lamb.'}]


In [8]:
# 1) Load final RAW-C file so we only keep stimuli that are in the final dataset
rawc = pd.read_csv("/content/ChickenHotWings/data/processed/raw-c.csv")

# Build the set of final (Word, String) pairs
final_pairs = (
    rawc[["word", "string"]]
    .drop_duplicates()
    .rename(columns={"word": "Word", "string": "String"})
)

# Use the already-created dataframe from your previous cell: `data`
final_data = (
    data.merge(final_pairs, on=["Word", "String"], how="inner")
    .reset_index(drop=True)
)

print("Final stimulus-level data:")
print(final_data.head())
print(f"Number of final stimuli: {len(final_data)}")

Final stimulus-level data:
      String       Word                            M1_a  \
0       lamb       lamb  They liked the marinated lamb.   
1       book       book     He had a best-selling book.   
2  breakfast  breakfast   They ate a pancake breakfast.   
3    chicken    chicken   They liked the juicy chicken.   
4      class      class      It was a perceptive class.   

                               M1_b                              M2_a  \
0      They liked the grilled lamb.         They liked the cute lamb.   
1            He had a popular book.              He had a heavy book.   
2  They ate a nutritious breakfast.      They ate a family breakfast.   
3   They liked the roasted chicken.  They liked the clucking chicken.   
4       It was a misbehaving class.            It was a boring class.   

                            M2_b  
0  They liked the friendly lamb.  
1   He had a leather-bound book.  
2   They ate a lonely breakfast.  
3    They liked the pet chicken.  
4   

In [9]:
# 2) Reshape so each row is one target word in one sentence
long_data = (
    final_data
    .melt(
        id_vars=["Word", "String"],
        value_vars=["M1_a", "M1_b", "M2_a", "M2_b"],
        var_name="meaning_id",
        value_name="sentence"
    )
    .reset_index(drop=True)
)

# Rename for clarity:
# Word   = lemma / regular form
# String = lexeme / surface form in the sentence
long_data = long_data.rename(columns={"Word": "lemma", "String": "word"})

print("\nSentence-level data to embed:")
print(long_data.head(8))
print(f"Number of sentence instances: {len(long_data)}")


Sentence-level data to embed:
          lemma          word meaning_id                            sentence
0          lamb          lamb       M1_a      They liked the marinated lamb.
1          book          book       M1_a         He had a best-selling book.
2     breakfast     breakfast       M1_a       They ate a pancake breakfast.
3       chicken       chicken       M1_a       They liked the juicy chicken.
4         class         class       M1_a          It was a perceptive class.
5  contribution  contribution       M1_a  He made a charitable contribution.
6        design        design       M1_a       They liked the floral design.
7        dinner        dinner       M1_a       They liked the turkey dinner.
Number of sentence instances: 448


In [10]:
# 3) Optional small test first
test_n = 8
test_output = model.embed(
    words=long_data["word"].head(test_n).tolist(),
    target_texts=long_data["sentence"].head(test_n).tolist(),
    layers_id=[4, 8, 12],
    batch_size=8,
    return_static=True
)

print("\nSmall test output keys:")
if isinstance(test_output, dict):
    print(list(test_output.keys()))
else:
    print(type(test_output), test_output)

Text tokenization:   0%|          | 0/8 [00:00<?, ? examples/s]

  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 1/1 [00:04<00:00,  4.81s/it]


Small test output keys:
[4, 8, 12, -1]


In [13]:
# 4) Full embedding extraction
embed_output = model.embed(
    words=long_data["word"].tolist(),
    target_texts=long_data["sentence"].tolist(),
    layers_id=[4, 8, 12],
    batch_size=32,   # lower to 16 if Colab runs out of memory
    return_static=True
)


  7%|▋         | 1/14 [01:03<13:49, 63.82s/it]


Text tokenization:   0%|          | 0/448 [00:00<?, ? examples/s]

100%|██████████| 14/14 [03:45<00:00, 16.10s/it]


In [14]:
# 5) Attach embeddings robustly, regardless of exact output key names
result = long_data.copy()

if not isinstance(embed_output, dict):
    raise TypeError(f"Expected model.embed(...) to return a dict, got {type(embed_output)}")

for key, value in embed_output.items():
    key_str = str(key).lower()

    if key_str == "static":
        col_name = "embedding_static"
    else:
        digits = "".join(ch for ch in key_str if ch.isdigit())
        col_name = f"embedding_{digits}" if digits else f"embedding_{key_str}"

    arr = np.asarray(value)
    if arr.shape[0] != len(result):
        raise ValueError(f"{col_name} has {arr.shape[0]} rows, expected {len(result)}")

    result[col_name] = list(arr)
    print(f"Added {col_name}: {arr.shape}")

# Keep the required columns in a clean order
required_cols = [
    "lemma",          # regular form
    "word",           # surface form in sentence
    "meaning_id",
    "sentence",
    "embedding_static",
    "embedding_4",
    "embedding_8",
    "embedding_12",
]

result = result[required_cols]


Added embedding_4: (448, 768)
Added embedding_8: (448, 768)
Added embedding_12: (448, 768)
Added embedding_1: (448, 768)


KeyError: "['embedding_static'] not in index"

In [ ]:
# 6) Save to pickle
pickle_path = "/content/rawc_homonym_embeddings.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(result, f)

In [ ]:

# 7) Load again and print to verify
with open(pickle_path, "rb") as f:
    loaded_result = pickle.load(f)

print("\nReloaded pickle preview:")
print(loaded_result.head())
print(f"\nSaved pickle path: {pickle_path}")
print(f"Total embedded rows: {len(loaded_result)}")

In [ ]:
# 8) Check and print the entry 'bank'
# Try lemma first, then surface form
bank_rows = loaded_result[
    (loaded_result["lemma"].astype(str).str.lower() == "bank") |
    (loaded_result["word"].astype(str).str.lower() == "bank")
]

print("\nData for 'bank':")
print(bank_rows)


In [ ]:
# Optional: show embedding shapes for bank
if len(bank_rows) > 0:
    for col in ["embedding_static", "embedding_4", "embedding_8", "embedding_12"]:
        print(f"{col} shapes:", bank_rows[col].apply(lambda x: np.asarray(x).shape).tolist())

# Task 2: Compute sense embeddings for the homonym dataset using WordNet [20 points]

Your next task is to fetch the definitions (glosses) of the homonyms, and compute an embedding for each gloss (each gloss is associated with a specific sense). We do that so we can later see whether the contextualised embeddings computed above represent the meaning of the homonym in context well (by comparing it to the sense embeddings). Figure 18.9 in [Jurafsky's and Martin's (2021) chapter 18](https://web.stanford.edu/~jurafsky/slp3/old_sep21/18.pdf) graphically illustrates this idea. Use this chapter for this part of the assignment, as it will come useful for you both theoretically and practically.

## Task 2.1: Fetch senses and glosses for a word [5 points]

First of all, you will have to figure out how [WordNet](https://www.nltk.org/howto/wordnet.html) works within the nltk package (hint: pay attention to what a synset is).

Install and import all the necessary components and define a function to extract the glosses of a word and create a dictionary with senses and glosses.

Then use the word "bat" to test that everything is working correctly: i.e., for "bat", you should be able to get its senses and the gloss for each of the sense (you will see that synsets might contain related words, but you only need the senses that contain the word of interest or derivates thereof; this should be specified in the function). Print the output for "bat".


In [ ]:
#imports for task 2
from nltk.corpus import wordnet as wn
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
#example synset of bat
wn.synsets('bat')

[Synset('bat.n.01'),
 Synset('bat.n.02'),
 Synset('squash_racket.n.01'),
 Synset('cricket_bat.n.01'),
 Synset('bat.n.05'),
 Synset('bat.v.01'),
 Synset('bat.v.02'),
 Synset('bat.v.03'),
 Synset('bat.v.04'),
 Synset('cream.v.02')]

For my own clarity:

A **sense** is one meaning of a word. There are multiple senses for mouse: the animal and the computer item.

A **gloss** is the definition of the sense, so basically the corresponding dictionary entry for the context.

This means that for each word, there can be multiple senses, and for each sense, there is one gloss.

In [ ]:
# test block
[str(lemma.name()) for lemma in wn.synset('bat.n.05').lemmas()]

['bat']

In [ ]:
def extract_glosses(word, dict_only=True):
  """
  Extracts glosses for a given word.
  Input: word --> string representing a word
         dict_only --> boolean indicating whether to ignore the print statements or not
  Returns: dictionary containing senses and glosses
  """
  glosses_dict = dict()
  #for each synset of the input word
  for synset in wn.synsets(word):
    if dict_only is False:
      print(synset)
      if len(synset.examples()) > 0:                    #if there are examples
        print('examples: ', synset.examples())          #print the examples
      #the lemma represents a specific sense of a specific word
      print('lemmas: ', [str(lemma.name()) for lemma in synset.lemmas()])     #print the lemma (always existing)
      print('gloss:', synset.definition(), '\n')        #print the gloss
    glosses_dict[synset] = synset.definition()
  return glosses_dict

In [ ]:
extract_glosses(word='bat') # <-- add dict_only=False to view additional information

{Synset('bat.n.01'): 'nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate',
 Synset('bat.n.02'): '(baseball) a turn trying to get a hit',
 Synset('squash_racket.n.01'): 'a small racket with a long handle used for playing squash',
 Synset('cricket_bat.n.01'): 'the club used in playing cricket',
 Synset('bat.n.05'): 'a club used for hitting a ball in various games',
 Synset('bat.v.01'): 'strike with, or as if with a baseball bat',
 Synset('bat.v.02'): 'wink briefly',
 Synset('bat.v.03'): 'have a turn at bat',
 Synset('bat.v.04'): 'use a bat',
 Synset('cream.v.02'): 'beat thoroughly and conclusively in a competition or fight'}

## Task 2.2: Function to compute sense embeddings [10 points]

Now that you have a function to extract senses and glosses for a given word, write a function that takes a word and computes embeddings for each of the senses following the method explained in Jurafsky's and Martin's chapter. In this case, no need to calculate at different layers: you should use the last layer only. You should maximally optimise this function like before.

The output should include the sense, the gloss, and the embedding. Print the function's output when using the word "bank".


In [ ]:
def compute_embeddings(word):
  """
  Input: word --> string representing a word
  Returns: the senses, glosses and embeddings for the input word
  """
  #obtain glosses using the function we made earlier
  glosses = extract_glosses(word)

  ##TODO: compute embeddings for each sense
  #from the given description in chapter 18, I do not understand how sense embeddings are different when they belong to the same word?



## Task 2.3: Compute sense embeddings for the RAW-C stimuli [5 points]

Now, use the function you defined above to compute sense embeddings for the RAW-C stimuli and pickle dump it too.

As above, the information that should be there for each word is: the sense, the gloss, the embedding at the last layer. Again, you can think of which structure to use best, but keep in mind that we will have to compare these to the CWE calculated in task 1, so it is good to think of a similar structure that is easily comparable.

Make sure that the number of stimuli matches the number of stimuli in the final RAW-C dataset.

# Task 3: Compute and explore similarity between homonym CWEs and sense embeddings [35 points]

You now have the homonym CWEs computed in task 1, and the sense embeddings computed in task 2. The next step is to calculate cosine similarities between each CWE for each homonym (at the selected layer!) and each sense embedding for that homonym.

For instance, say for the word "bat" with meaning M1_a, you have its CWE at the static layer and at layers 4, 8, 12 and 7 senses: here, you will end up with 16 cosine similarities (take each CWE and compute its similarity to each of the sense embeddings). We then want to see which sense meaning is the closest to each CWE, and do some qualitative explorations with that.

## Task 3.1: Compute the cosine similarity between all the CWEs and the sense embeddings [8 points]

This task is not trivial with regards to how much information you have and how to structure the data (this is why it's also important to think of data structures in the earlier parts of the assignment), so take some time to think how to best breakdown this task. Test each step/function if you have multiple. Pickle dump your final output so that it is easily retrievable for later. At the end, print an example of the entry "bank".

For cosine similarity, the cdist function from scipy.spatial.distance seems the most efficient, but you are free to use any of your liking (hint: pay attention to the shape of your embeddings and to similarity vs distance. You will need the similarity).

In [ ]:
#your code here

## Task 3.2: Quantitative and qualitative explorations the relationship between homonym embeddings and dominant senses

Now, we can look into how the CWEs in different meanings and layers relate to the different senses of a homonym. We'll focus on the dominant sense in WordNet, see below for more details. This section includes both code blocks and reflection questions.

### Dominant senses in WordNet and top senses across layers (focus on static layer) [8 points]

Embeddings at the static layer do not take into account context, so intuitively they should capture the 'average' meaning, maybe the most common/dominant. We can test this by looking at the most similar sense and seeing if that matches that most common/dominant sense in the synset.

Keep in mind that synsets mark more common/dominant senses with numbering: so n.01 will be the most common noun; v.01 the most common verb, etc. If that is not available, the most common meaning will be the next number (e.g., n.02). You have to take that into account when you extract the top sense, so first extract information about which are the most dominant senses for each word across all the parts of speech: for example, "bat" might have as its two most common senses bat.n.01 and bat.v.02 (because v.01 might not be available; this is just an example). Some words might only have one part of speech in their synset, some more. Print your results.

In [ ]:
#your code here

Then, extract the top similarity of homonyms to the senses at all the layers you have available. While we are interested in the static layer for checking dominant senses, it is also interesting to look into other layers to see whether adding context will refine the captured meaning.


In [ ]:
#your code here

Let's check an example from our results.

Out of all the similarities of 'bank' to all its senses at all the layers, which one is the highest? Print your results for that entry and reflect below.

In [ ]:
#your code here

### Does the static layer capture the most dominant meaning, according to WordNet (and according to you)? [2 point]

%your answer here

### Across other layers and meanings, which layer seems to capture the meaning of bank across meanings best, and why do you make this conclusion? [2 points]

%your answer here

### Checking matches and mismatches with the dominant sense [5 points]

Now, let's quantitatively check if the static layer actually captures the most dominant sense (any POS). You should end up with two data structures: matches (when the most similar sense is one of the dominant senses) and mismatches (when the most similar sense is not one of the dominant sense). Do that also for the other layers to compare. Print the percentage of matches and mismatches per layer.



In [ ]:
#your code here

Now, print the matches and mismatches for the static layer only.

In [ ]:
#your code here

### Do BERT's static embeddings capture the most dominant sense in WordNet? [2 point]

%your answer here

### Do the percentages of matches and mismatches throughout the layers make sense to you or is it different than what you expected? [2 points]

%your answer here

### For the **static layer**, are there any words that seem to particularly deviate from the dominant meaning? If so, which and why could that be? [3 points]

%your answer here

### Do you think the corpus on which BERT is trained might reflect different meaning dominance than for WordNet's senses? If so/not, why? [3 points]

%your answer here

# Task 4: Partially replicate Trott & Bergen's experiment [20 points]

Now comes the time to partially replicate the RAW-C experiment, by seeing whether different layers of BERT capture meanings more or less similarly to humans. At the end you will have to wrap up with a brief comment on which layer seems to capture meanings best and how that connects to explorations in the previous section.

## Task 4.1: Create a dataframe with cosine similarities between sentences at different layers [7 points]

You should now use the embeddings at the different layers that you computed to calculate similarities between each context: M1a, M1b, M2a, M2b. You will have to have all combinations, so for each string in the RAW-C dataframe, you'll have: M1a vs M1b, M1a vs M2a, M1a vs M2b, M1b vs M2a, M1b vs M2b, M2a vs M2b.

Bear in mind that your final dataframe should include: the word, the string as it appears in the sentence, cosine similarity at layers 4, layer 8, layer 12, the version being compared (is it M1a vs M1b or M1a vs M2a?) and the mean relatadness given by humans (hint: the repo you cloned will come useful here, both in terms of code and data). Print the head of the dataframe to check everything is in order, and check also that the number of stimuli match with your number across the assignment (starting from task 1).

In [ ]:
#your code here

## Task 4.2: Correlate with human judgements and visualise [8 points]

First, correlate the cosine similarities at the different layers to the mean human relatedness judgements. Use the same correlation metric used by Trott & Bergen.

In [ ]:
#your code here

Next, visualise your results. You want to see the correlation between BERT embeddings and human judgements per layer, but what would also be interesting is to include the meaning contrasts (such as M1_a_M1_b, etc), so that we can see how those play out per layer.

In [ ]:
#your code here

### Reflect on the correlations and on the visualisations. What can you observe and infer in terms of which layer(s) might be capturing meaning best? Is there one way to determine that (i.e., what does 'capturing meanings' mean?)? Contrast and compare the layers. [5 points]

%your answer here



